# IOAI — 2024 First Stage Object Tracking (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
if not os.path.exists('data/level_1'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2024-first-stage-object-tracking/data.zip', 'd.zip')
    zipfile.ZipFile('d.zip').extractall('data')
print('레벨:', sorted(os.listdir('data')))
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# 물체 추적 — 세 컵 게임 (Object Tracking)

폴란드 AI 올림피아드 2024 (1차 예선). **세 개의 컵**이 섞이는 애니메이션에서, 각 프레임마다
탐지 모델이 내놓은 **경계 상자(bounding box)** 3개가 주어진다. 상자는 프레임마다 **순서가 뒤섞여**
제공되므로(리스트 위치 ≠ 컵 정체성), 프레임 간에 컵을 **추적(association)** 해서 마지막 프레임의
**왼→오 최종 순열**을 맞혀야 한다.

- 초기(첫 프레임) 왼→오 컵 정체성 = `[0, 1, 2]`.
- 목표 = 마지막 프레임에서 각 왼→오 위치에 있는 컵의 정체성, 예: `[0, 2, 1]`.
- 데이터 3레벨: **L1** 깨끗 · **L2** 가림/블러로 상자 누락 · **L3** 더 긴 시퀀스.
  (원본 L3 는 상자 없이 이미지에서 직접 검출하는 과제지만, 여기서는 세 레벨 모두 제공된
  모델 탐지 상자로 *추적 능력*에 집중한다.)

**제출**: `submission.csv` — `id,p0,p1,p2` (id=`L{레벨}_{영상번호:04d}`, 150행).
**채점**: 레벨별 정확도 → 배점(레벨당 0~0.5, 총 0~1.5). L1 0.5→0·1.0→0.5 / L2 0.5→0·0.95→0.5 / L3 0.3→0·1.0→0.5.


In [ ]:
# 데이터 준비 (Colab: 자동 다운로드 / DGX: data/ 이미 존재)
import os, urllib.request, zipfile
if not os.path.exists("data/level_1"):
    url = "https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/2024-first-stage-object-tracking/data.zip"
    urllib.request.urlretrieve(url, "d.zip"); zipfile.ZipFile("d.zip").extractall("data")
print("레벨:", sorted(os.listdir("data")))


In [ ]:
import json, csv, numpy as np

def load_coords(level, vid):
    """coordinates_XXXX.json -> {frame_name: [[xmin,ymin,xmax,ymax] x3]} (상자 순서는 프레임마다 뒤섞임)"""
    with open(f"data/level_{level}/coordinates/coordinates_{vid:04d}.json") as f:
        return json.load(f)

def center(box):
    return np.array([(box[0]+box[2])/2.0, (box[1]+box[3])/2.0])


In [ ]:
def your_algorithm(coordinates):
    """근접-이웃 다중물체 추적 + 속도 예측(가림 대응).

    아이디어: 컵은 프레임 사이 부드럽게 움직인다. 첫 (3개 상자) 프레임에서 왼->오로
    정체성 [0,1,2] 을 심고, 매 프레임 각 정체성의 '예측 위치(직전 위치+속도)' 에 가장 가까운
    상자를 탐욕적으로 1:1 배정한다. 상자가 누락(가림)되면 예측 위치로 코스팅한다.
    마지막 프레임의 정체성들을 왼->오로 정렬한 순서가 최종 순열."""
    frames = sorted(coordinates.keys())
    f0 = next(f for f in frames if len(coordinates[f]) == 3)
    boxes = coordinates[f0]
    order = sorted(range(3), key=lambda i: center(boxes[i])[0])
    pos = [center(boxes[order[k]]) for k in range(3)]
    vel = [np.zeros(2) for _ in range(3)]
    SMOOTH = 0.4
    for f in frames[frames.index(f0)+1:]:
        cur = [center(b) for b in coordinates[f]]
        pred = [pos[k] + vel[k] for k in range(3)]
        pairs = sorted((float(np.sum((pred[k]-cur[j])**2)), k, j)
                       for k in range(3) for j in range(len(cur)))
        ak, aj, newpos = set(), set(), [None]*3
        for _, k, j in pairs:
            if k in ak or j in aj:
                continue
            ak.add(k); aj.add(j)
            vel[k] = (1-SMOOTH)*vel[k] + SMOOTH*(cur[j]-pos[k])
            newpos[k] = cur[j]
        for k in range(3):
            if newpos[k] is None:
                newpos[k] = pos[k] + vel[k]
        pos = newpos
    return sorted(range(3), key=lambda k: pos[k][0])


In [ ]:
# 150개 영상 예측 -> submission.csv
rows = []
for level in (1, 2, 3):
    for vid in range(50):
        perm = your_algorithm(load_coords(level, vid))
        rows.append([f"L{level}_{vid:04d}", int(perm[0]), int(perm[1]), int(perm[2])])
with open("submission.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["id", "p0", "p1", "p2"]); w.writerows(rows)
print("submission.csv 저장:", len(rows), "행")


### 성능 (valid 150영상)
- L1 정확도 1.00 (0.50점) · L2 0.90 (0.44점) · L3 1.00 (0.50점) -> **총 ≈1.44 / 1.5**
- 베이스라인(초기순열 고정)은 0점. 핵심은 속도예측으로 **가림(L2 상자 누락)** 을 견디는 것.


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)